### BLOCK 1 / BLOCO 1: Headquarters Setup (Installation / Instalação)
* 🇧🇷 Neste bloco, instalamos o motor de orquestração multi-agente (CrewAI) e utilitários.
* 🇺🇸 In this block, we install the multi-agent orchestration engine (CrewAI) and tabular utilities.

# Lab 07: Operation Churn Zero (CrewAI Multi-Agent Squad + Gemini 3.8 Flash)
**Course / Disciplina:** Augmented Analytics & AI-Driven Insights (FIAP MBA)  
**[🇧🇷 PT-BR]** Orquestrar uma Squad de IAs especialistas para reverter cancelamentos de clientes em tempo real.  
**[🇺🇸 EN]** Orchestrate a specialized Multi-Agent AI Squad to mitigate high-value customer churn in real time.

---

### 🚨 WAR ROOM SCENARIO / CENÁRIO DE GUERRA
* 🇧🇷 A diretoria detectou uma anomalia na retenção de clientes Premium. Sua missão é ativar uma **Squad de Agentes de IA** que irá: (1) Identificar o cliente de maior risco; (2) Criar uma oferta de retenção personalizada; (3) Redigir um e-mail executivo de alto impacto.
* 🇺🇸 The board detected a retention anomaly among Premium accounts. Your mission is to deploy an **Autonomous AI Agent Squad** to: (1) Isolate the highest-risk customer; (2) Engineer a tailored retention offer; (3) Draft a high-impact executive outreach email.

In [ ]:
# Install CrewAI and utilities in the Colab environment
!pip install -q crewai openpyxl tabulate
print("✅ Environment configured successfully! / Ambiente configurado com sucesso!")

### BLOCK 2 / BLOCO 2: Central Command Connection (API Key & Model Routing)
* 🇧🇷 Configuramos a API Key, o idioma de saída e a string canônica LiteLLM (`gemini/gemini-3.8-flash`) para evitar erros de validação no Pydantic v2.
* 🇺🇸 Configure your API Key, output language, and canonical LiteLLM model string (`gemini/gemini-3.8-flash`) for stable Pydantic v2 validation.

In [ ]:
import os
import pandas as pd
from crewai import Agent, Task, Crew, Process

# 🇧🇷 Substitua 'COLE_SUA_API_KEY_AQUI' pela sua chave do Google AI Studio | 🇺🇸 Replace with your Google AI Studio API Key
API_KEY = "COLE_SUA_API_KEY_AQUI"
OUTPUT_LANGUAGE = "Portuguese (Brazil)" # @param ["Portuguese (Brazil)", "English"]
os.environ["GOOGLE_API_KEY"] = API_KEY
os.environ["GEMINI_API_KEY"] = API_KEY

# Canonical LiteLLM/CrewAI model string for Google Gemini 3.8 Flash
llm_config = "gemini/gemini-3.8-flash"

print(f"✅ Configured model / Modelo configurado: {llm_config} | Output: {OUTPUT_LANGUAGE}")

### BLOCK 3 / BLOCO 3: Data Intelligence Ingestion (Self-Healing)
* 🇧🇷 Carregamento resiliente do dataset de evasão e extração da amostra VIP (top 5 registros).
* 🇺🇸 Resilient self-healing ingestion of the churn dataset and extraction of the VIP cohort sample (top 5 records).

In [ ]:
import os
import urllib.request

DATASET_NAME = 'Lab 05 - EvasaoClientesClassificados.xlsx'
RAW_URL = 'https://raw.githubusercontent.com/rafaelmatsuyama/FIAP-BI-AugmentedAnalytics/main/lab05-automl-scikit/Lab%2005%20-%20EvasaoClientesClassificados.xlsx'

if not os.path.exists(DATASET_NAME):
    print(f"📥 Downloading / Baixando {DATASET_NAME} from official repository...")
    urllib.request.urlretrieve(RAW_URL, DATASET_NAME)

try:
    df = pd.read_excel(DATASET_NAME)

    # Extract top 5 VIP records serialized in Markdown for surgical agent deliberation
    amostra_clientes = df.head(5).to_markdown()
    print("✅ Data intelligence loaded. Squad ready for briefing! / Inteligência de dados carregada!")
except Exception as e:
    print(f"❌ ERROR loading dataset / ERRO ao carregar dataset: {e}")

### BLOCK 4 / BLOCO 4: Assembling the Squad (Agent Definitions)
* 🇧🇷 Instanciamos 3 agentes especialistas com papéis, metas e backstories distintos.
* 🇺🇸 Instantiate 3 specialized agents with distinct roles, goals, and backstories.

In [ ]:
investigador = Agent(
    role='Senior Churn Data Investigator (The Investigator)',
    goal=f'Identify risk patterns across the 5 provided customer records and isolate the highest-risk account (respond in {OUTPUT_LANGUAGE}).',
    backstory='You are a seasoned forensic data detective specialized in diagnosing the exact drivers of customer attrition.',
    allow_delegation=False,
    llm=llm_config
)

arquiteto = Agent(
    role='Retention Strategy Architect (The Architect)',
    goal=f'Design a tailored, high-ROI retention offer calibrated to the customer coverage tier (respond in {OUTPUT_LANGUAGE}).',
    backstory='You are a master commercial strategist who engineers retention packages that protect margin while reversing churn.',
    allow_delegation=False,
    llm=llm_config
)

comunicador = Agent(
    role='Executive Customer Success Communicator (The Communicator)',
    goal=f'Write an empathetic, executive-grade retention email in {OUTPUT_LANGUAGE} ready to be sent to the customer.',
    backstory='You are a corporate communications specialist skilled at turning cold analytical insights into high-converting executive outreach.',
    allow_delegation=False,
    llm=llm_config
)
print("✅ Squad assembled: Investigator, Architect, and Communicator online! / Squad convocada!")

### BLOCK 5 / BLOCO 5: Action Plan (Sequential Task Chaining)
* 🇧🇷 Definimos tarefas sequenciais onde o output de um agente alimenta o próximo (*Chain-of-Thought*).
* 🇺🇸 Define sequential tasks where each agent's output feeds the next specialist (*Chain-of-Thought*).

In [ ]:
tarefa_analise = Task(
    description=f"Analyze these customer records and isolate the account with the highest churn risk: {amostra_clientes}",
    expected_output=f"A technical diagnostic summary in {OUTPUT_LANGUAGE} identifying the most critical customer and root risk drivers.",
    agent=investigador
)

tarefa_estrategia = Task(
    description="Based on the Investigator's diagnosis, design a compelling tier upgrade or commercial retention package.",
    expected_output=f"A detailed retention strategy in {OUTPUT_LANGUAGE} tailored to the critical customer profile.",
    agent=arquiteto
)

tarefa_email = Task(
    description="Draft the final executive retention outreach email using a warm, VIP advisory tone.",
    expected_output=f"The complete outreach email in {OUTPUT_LANGUAGE} formatted in clean Markdown ready for dispatch.",
    agent=comunicador
)
print("✅ Action plan locked. Ready to launch sequential orchestration! / Plano de ação traçado!")

### BLOCK 6 / BLOCO 6: Launching the Crew (Execution)
* 🇧🇷 Disparamos o pipeline sequencial e inspecionamos a trilha de raciocínio (*Thought*) de cada agente.
* 🇺🇸 Trigger the sequential pipeline and inspect each agent's reasoning trace (*Thought*) in the execution logs.

In [ ]:
squad_retencao = Crew(
    agents=[investigador, arquiteto, comunicador],
    tasks=[tarefa_analise, tarefa_estrategia, tarefa_email],
    process=Process.sequential,
    verbose=True
)

print("\n🚀 SQUAD IN ACTION! Waiting for operation completion... / SQUAD EM AÇÃO!\n")
resultado_final = await squad_retencao.kickoff_async()

print("\n" + "="*80)
print("📄 OPERATION CHURN ZERO FINAL REPORT / RELATÓRIO FINAL DA OPERAÇÃO CHURN ZERO")
print("="*80 + "\n")
print(resultado_final)